## The task of this Notebook is to get pointed to a Data_year folder and process the data inside
### either by using a 20 day window or a 10 day window
### the processed data is than put either into the 20 or the 10 day data folder

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import  math
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import re
import argparse

### Here every county is merged to one coherent .csv file that has all data, this goes to Data_Fused_year 
Data_year folder can is delted after theire is a backup somewhere else

In [13]:
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter(action="ignore", category=PerformanceWarning)

# Canonical band schema = HLS-Landsat names. HLS-Sentinel exports name NIR/SWIR1/SWIR2
# as B8A/B11/B12; normalising here (BEFORE fusing tiles of a county) means Landsat and
# Sentinel observations land in the SAME column, so the later per-county weighted means
# are computed over the complete set of observations instead of being split.
# HLSS = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12', 'NDVI', 'GCVI']
# HLSL =['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'NDVI', 'GCVI']
BAND_RENAME = {"B8A": "B5", "B11": "B6", "B12": "B7"}

def normalize_band_columns(df):
    # Drop cloud-fraction stats
    cloud = [c for c in df.columns if c.startswith("cloud_fraction")]
    if cloud:
        df = df.drop(columns=cloud)

    rename = {}
    drop_cols = []

    for col in df.columns:
        prefix, sep, rest = col.partition("_")

        if sep and prefix in BAND_RENAME:
            target = f"{BAND_RENAME[prefix]}{sep}{rest}"
            rename[col] = target

            # Target already exists, so drop Sentinel duplicate
            drop_cols.append(col)

    if drop_cols:
        df = df.rename(columns=rename)
    return df
# Example:
# Harvester_2019_4_11_T15SXA_20190523T164207.csv
def parse_file(path: Path, target_year: int):
    FILE_RE =  re.compile(r"^Harvester_(?P<year>\d{4})_(?P<county_id>.+?)_(?P<tile>T[A-Z0-9]{5})_(?P<timestamp>\d{8}T\d{6,8})\.csv$")
    match = FILE_RE.match(path.name)
    if not match:
        return None

    year = int(match.group("year"))
    if year != target_year:
        return None

    county_id = match.group("county_id")
    timestamp = match.group("timestamp")

    date_part = timestamp[:8]
    dt = datetime.strptime(date_part, "%Y%m%d")

    return {
        "path": path,
        "county_id": county_id,
        "timestamp": timestamp,
        "date": f"{dt.day}.{dt.month}",
    }


def fuse_county(county_id: str, file_infos: list[dict], output_folder: Path, year: int):
    dfs = []

    # Sort by date so the output is deterministic
    file_infos = sorted(file_infos, key=lambda x: x["timestamp"])

    for info in file_infos:
        df = pd.read_csv(info["path"])
        df = normalize_band_columns(df)   # canonicalise bands before fusing

        # Add/overwrite date column
        df["date"] = info["date"]

        dfs.append(df)

    fused = pd.concat(dfs, ignore_index=True)

    output_path = output_folder / f"Harvester_{year}_{county_id}.csv"
    fused.to_csv(output_path, index=False)

    return output_path, len(fused), len(file_infos)


def fuse_harvester_files(basedir: str, year: int, workers: int = 10, indir: str = "Data_"):
    basedir = Path(basedir)

    input_folder = basedir / (indir+str(year)) 
    print(input_folder)
    output_folder = basedir / (indir+"Fused_"+str(year)) 
    output_folder.mkdir(parents=True)

    grouped = {}

    for path in input_folder.glob("Harvester_*.csv"):
        info = parse_file(path, year)
        if info is None:
            continue

        grouped.setdefault(info["county_id"], []).append(info)

    print(f"Found {sum(len(v) for v in grouped.values())} files")
    print(f"Found {len(grouped)} counties")
    print(f"Output folder: {output_folder}")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [
            executor.submit(fuse_county, county_id, infos, output_folder, year)
            for county_id, infos in grouped.items()
        ]

        for future in as_completed(futures):
            output_path, rows, source_files = future.result()
            print(f"Created {output_path.name}: {rows} rows from {source_files} files")


#fuse_harvester_files(basedir=".", year=2025, workers=50)


### Here every county is processed to one coherent .csv file this goes to Data_year_ai
Only the mean values of every 10/20 slot are saved for the entire county
The data in these folders are than used for data science and ML


In [12]:
def weighted_county_std(cell_means, cell_stds, weights):
    valid = cell_means.notna() & cell_stds.notna() & weights.notna() & (weights > 0)
    if not valid.any():
        print("end=> No valid data for county, returning NaN")
        return np.nan
    county_mean = np.average(cell_means[valid], weights=weights[valid])
    second_moment = np.average(cell_stds[valid] ** 2 + (cell_means[valid]-county_mean) ** 2, weights=weights[valid])
    return np.sqrt(max(second_moment, 0.0))


# Example usage:
# Group 1: N=30, Mean=50, SD=5
# Group 2: N=40, Mean=55, SD=6
result = weighted_county_std(pd.Series([11.8,15.3, 8.4], copy=False), pd.Series([2.4,3.2, 4.1], copy=False), pd.Series([10, 20, 15], copy=False))
print(result)
print(f"Combined Standard Deviation:" + str(result))


4.533646503344074
Combined Standard Deviation:4.533646503344074


In [14]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
import pandas as pd
import numpy as np


DROP_COLUMNS = {"image_id", "longitude", "latitude", "date"}


def parse_day_month(value: str, dummy_year: int = 2000) -> date:
    # date format from your created files: "23.5", "9.4", etc.
    s = str(value).strip()
    day, month = s.split(".")
    return date(dummy_year, int(month), int(day))


def format_day_month(d: date) -> str:
    return f"{d.day}.{d.month}"


def make_windows(start_dm: str, end_dm: str, frequency_days: int):
    start = parse_day_month(start_dm)
    end = parse_day_month(end_dm)

    windows = []
    current = start

    while current + timedelta(days=frequency_days - 1) <= end:
        w_start = current
        w_end = current + timedelta(days=frequency_days - 1)
        windows.append((w_start, w_end))
        current = w_end + timedelta(days=1)

    return windows


def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")

    valid = values.notna() & weights.notna() & (weights > 0)

    if not valid.any():
        return np.nan

    return np.average(values[valid], weights=weights[valid])


def get_county_id_from_filename(path: Path) -> str:
    # Example: Harvester_2019_4_11.csv -> 4_11
    parts = path.stem.split("_")
    return "_".join(parts[2:])


def fill_missing_isolated_windows(out: pd.DataFrame, feature_cols: list[str]):
    """
    Fill only isolated missing windows:
    real, missing, real -> missing = avg(previous, following)

    from_data codes:
      1 = calculated from real data
      3 = filled as mean of previous and following window
      0 = missing / unfilled, should not remain in final output

    If missing is first/last or if multiple missing windows are adjacent,
    raise ValueError.
    """

    for i in range(len(out)):
        # 1 = real data, 3 = already filled
        if out.at[i, "from_data"] != 0:
            continue

        if i == 0 or i == len(out) - 1:
            raise ValueError("missing edge window")

        if out.at[i - 1, "from_data"] == 0 or out.at[i + 1, "from_data"] == 0:
            raise ValueError("consecutive missing windows")

        out.loc[i, feature_cols] = (
            out.loc[i - 1, feature_cols].astype(float)
            + out.loc[i + 1, feature_cols].astype(float)
        ) / 2.0

        # 3 = this window was filled from neighboring windows
        out.at[i, "from_data"] = 3

    if (out["from_data"] == 0).any():
        raise ValueError("unfilled missing windows remain")

    return out


def process_one_county_file(
    input_path: Path,
    output_folder: Path,
    frequency_days: int,
    start_dm: str = "4.7",
    end_dm: str = "31.8",
):
    output_path = output_folder / input_path.name

    if output_path.exists():
        print(f"{input_path.name} already processed, skipping")
        return input_path.name, None

    county_id = get_county_id_from_filename(input_path)

    df = pd.read_csv(input_path)

    if df.empty:
        raise ValueError("empty file")

    if "date" not in df.columns:
        raise ValueError("missing date column")

    df["_parsed_date"] = df["date"].apply(parse_day_month)

    count_cols = [c for c in df.columns if c.endswith("_count")]

    if not count_cols:
        raise ValueError("missing *_count columns")

    # All count columns are identical per row, so one is enough.
    weight_col = count_cols[1]
    feature_cols = [
        c for c in df.columns
        if c not in DROP_COLUMNS
        and c != "_parsed_date"
        and not c.endswith("_count")
    ]

    if not feature_cols:
        raise ValueError("no feature columns left after dropping columns")

    windows = make_windows(start_dm, end_dm, frequency_days)

    rows = []

    for w_start, w_end in windows:
        window_df = df[
            (df["_parsed_date"] >= w_start)
            & (df["_parsed_date"] <= w_end)
        ]

        row = {
            "date_start": format_day_month(w_start),
            "date_end": format_day_month(w_end),
            "from_data": 1 if len(window_df) > 0 else 0,
        }

        if len(window_df) == 0:
            for col in feature_cols:
                row[col] = np.nan
        else:
            weights = window_df[weight_col]

            for col in feature_cols:
                sd_columns = ['B2_stdDev', 'B3_stdDev', 'B4_stdDev', 'B5_stdDev', 'B6_stdDev', 'B7_stdDev', 'NDVI_stdDev', 'GCVI_stdDev']
                if col in sd_columns:
                    row[col] = weighted_county_std(window_df[col.replace("_stdDev", "_mean")], window_df[col], weights)
                else:
                    row[col] = weighted_mean(window_df[col], weights)

        rows.append(row)

    out = pd.DataFrame(rows)

    # Fill missing windows using avg(previous, following).
    out = fill_missing_isolated_windows(out, feature_cols)

    # Important:
    # Do NOT reject all remaining NaNs here.
    # Some real-data columns like skew/kurtosis can be NaN from the source.
    # Missing-window validity is now checked only through from_data.
    if (out["from_data"] == 0).any():
        raise ValueError("unfilled missing windows remain")

    out.to_csv(output_path, index=False)

    return input_path.name, None


def compress_harvester_files(year: int, frequency_days: int, start_dm: str = "7.4", end_dm: str = "31.8", workers: int = 10):
    basedir = Path(".")
    input_folder = basedir / f"Data_Fused_{year}"
    output_folder = basedir / f"Data_Windowed_{year}_{frequency_days}d"

    # Error if folder already exists.
    output_folder.mkdir(parents=True, exist_ok=False)

    files = sorted(input_folder.glob("Harvester_*.csv"))

    if not files:
        raise FileNotFoundError(f"No Harvester_*.csv files found in {input_folder}")

    skipped = []

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {
            executor.submit(
                process_one_county_file,
                path,
                output_folder,
                frequency_days,
                start_dm,
                end_dm,
            ): path
            for path in files
        }

        for future in as_completed(futures):
            path = futures[future]

            try:
                filename, _ = future.result()
                print(f"Created {filename}")
            except Exception as e:
                skipped.append((path.name, str(e)))
                print(f"Skipped {path.name}: {e}")

    print("\nSkipped counties/files:")
    if skipped:
        for filename, reason in skipped:
            print(f"  {filename}: {reason}")
    else:
        print("  None")

    return skipped

## Procesing 20 day windows

In [6]:
years= [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]
year = [2025]
for i in years:
    compress_harvester_files(
        year=i,
        frequency_days=20,
        workers=40,
    )
    compress_harvester_files(
        year=i,
        frequency_days=10,
        workers=40,
    )

Skipped Harvester_2016_14_56.csv: missing edge window
Skipped Harvester_2016_14_17.csv: consecutive missing windows
Created Harvester_2016_14_40.csv
Created Harvester_2016_14_28.csv
Created Harvester_2016_14_81.csv
Created Harvester_2016_14_63.csv
Created Harvester_2016_14_7.csv
Created Harvester_2016_14_2.csv
Created Harvester_2016_14_48.csv
Skipped Harvester_2016_14_45.csv: missing edge window
Created Harvester_2016_14_42.csv
Created Harvester_2016_14_13.csv
Created Harvester_2016_14_30.csv
Created Harvester_2016_14_12.csv
Created Harvester_2016_14_75.csv
Skipped Harvester_2016_14_14.csv: missing edge window
Created Harvester_2016_14_25.csv
Created Harvester_2016_14_59.csv
Created Harvester_2016_14_43.csv
Created Harvester_2016_14_37.csv
Created Harvester_2016_14_21.csv
Created Harvester_2016_14_31.csv
Created Harvester_2016_14_54.csv
Created Harvester_2016_14_10.csv
Created Harvester_2016_14_86.csv
Created Harvester_2016_14_24.csv
Created Harvester_2016_14_90.csv
Created Harvester_2

In [15]:
# Repair the windows the unmasked NaN bug left without a standard deviation:
# reprocess only the county files that have an SD missing while the mean exists
BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "NDVI", "GCVI"]
basedir = Path(".")
for frequency_days in [20, 10]:
    for output_path in sorted(basedir.glob(f"Data_Windowed_*_{frequency_days}d/Harvester_*.csv")):
        windows = pd.read_csv(output_path)
        is_affected = False
        for band in BANDS:
            if (windows[f"{band}_stdDev"].isna() & windows[f"{band}_mean"].notna()).any():
                is_affected = True
        if not is_affected:
            continue
        year = int(output_path.stem.split("_")[1])
        input_path = basedir / f"Data_Fused_{year}" / output_path.name
        output_path.unlink()
        process_one_county_file(input_path, output_path.parent, frequency_days, start_dm="7.4", end_dm="31.8")
        print(f"reprocessed {output_path.parent.name}/{output_path.name}")

reprocessed Data_Windowed_2016_20d/Harvester_2016_23_57.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_23_66.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_23_80.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_28_15.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_34_8.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_35_2.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_35_28.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_36_19.csv
reprocessed Data_Windowed_2016_20d/Harvester_2016_50_62.csv
reprocessed Data_Windowed_2017_20d/Harvester_2017_35_18.csv
reprocessed Data_Windowed_2017_20d/Harvester_2017_35_5.csv
reprocessed Data_Windowed_2017_20d/Harvester_2017_48_20.csv
reprocessed Data_Windowed_2017_20d/Harvester_2017_48_36.csv
reprocessed Data_Windowed_2017_20d/Harvester_2017_6_32.csv
reprocessed Data_Windowed_2018_20d/Harvester_2018_34_18.csv
reprocessed Data_Windowed_2018_20d/Harvester_2018_34_36.csv
reprocessed Data_Windowed_2018_20d/Harvester